## LABORATORIO 6 Árbol de Expansión Mínima (MST)

Nuria Arroyo Bustamante & Erick Walter Trejo

Apredizaje de Máquina  - Otoño 2025

Dr. Hector Saib Gómez Maravillo 


## Instrucciones

### Parte uno

1. Selecciona un conjunto de datos adecuado (puede ser el mismo conjunto utilizado en los laboratorios anteriores o el correspondiente a tu proyecto final).  
2. Calcula una matriz de similitud/disimilitud de los datos seleccionados.  
3. Construye una gráfica completa ponderada en Networkx o igraph, donde cada vértice representa una instancia y el peso de cada arista corresponde a la medida de similitud/disimilitud entre instancias.  
4. Usando la gráfica completa del punto 3, obtén su Árbol de Expansión Mínima (MST), considerando los pesos como criterio.  
5. Aplica una técnica de reducción de dimensionalidad basada en la matriz de distancia para proyectar las instancias en R².  
6. Visualiza las instancias proyectadas y agrega a la visualización las aristas del MST en la misma gráfica.  
   **Nota:** Recuerda que el MST se construye a partir de la matriz de disimilitud original, no de las distancias resultantes de la reducción de dimensionalidad.


In [30]:
import pandas as pd
import numpy as np
import networkx as nx
import plotly.graph_objects as go
import umap.umap_ as umap
from sklearn.manifold import MDS, TSNE
import plotly.express as px


In [31]:
#creo que podemos utilizar la base de datos de moleculas que yo ya venia trabajando  
#leer la matriz de similitud del csv del lab 2 y crear la de disimilitud
matriz_similitud = pd.read_csv('matriz_similitud_wl3.csv', index_col=False)
matriz_similitud = matriz_similitud.to_numpy()
D = 1 - matriz_similitud

In [32]:
#parte 3
n = D.shape[0]
labels = [f"i={i}" for i in range(n)]


#grafo completo ponderado   
G = nx.Graph()
for i in range(n):
    for j in range(i+1, n):
        G.add_edge(i, j, weight=float(D[i, j]))

# MST
MST = nx.minimum_spanning_tree(G, weight='weight')


In [33]:

# parte 4,5,6
# umbral para separar aristas la media 
weights = np.array([d['weight'] for _, _, d in MST.edges(data=True)])
umbral = float(np.median(weights))

# Mostrar aristas removidas
edges_kept = [(u, v) for u, v, d in MST.edges(data=True) if d['weight'] <= umbral]
edges_removed = [(u, v) for u, v, d in MST.edges(data=True) if d['weight'] > umbral]

#MDS
random_state = 42
mds = MDS(n_components=2, dissimilarity='precomputed', random_state=random_state)
coords = mds.fit_transform(D)  # shape (n, 2)

def xy_from_edges(edges):
    xs, ys = [], []
    for i, j in edges:
        xs.extend([coords[i, 0], coords[j, 0], None])
        ys.extend([coords[i, 1], coords[j, 1], None])
    return xs, ys

x_kept, y_kept = xy_from_edges(edges_kept)
x_removed, y_removed = xy_from_edges(edges_removed)

# 7) Fig de Plotly
fig = go.Figure()

# Líneas del MST conservadas
fig.add_trace(go.Scatter(
    x=x_kept, y=y_kept,
    mode="lines",
    line=dict(width=2),
    opacity=0.9,
    name=f"MST (≤ umbral={umbral:.4g})"
))

show_removed = True
# Líneas removidas (opcional)
if show_removed and len(edges_removed) > 0:
    fig.add_trace(go.Scatter(
        x=x_removed, y=y_removed,
        mode="lines",
        line=dict(width=1, dash="dash"),
        opacity=0.6,
        name=f"Aristas removidas (> {umbral:.4g})"
    ))

# Puntos (instancias)
fig.add_trace(go.Scatter(
    x=coords[:, 0], y=coords[:, 1],
    mode="markers",
    marker=dict(size=10),
    text=labels,
    hovertemplate="<b>%{text}</b><br>X=%{x:.3f}<br>Y=%{y:.3f}<extra></extra>",
    name="Instancias"
))

fig.update_layout(
    title="Proyección 2D (MDS) con MST",
    xaxis_title="X",
    yaxis_title="Y",
    width=800,
    height=700,
    template="plotly_white",
    legend=dict(bgcolor="rgba(255,255,255,0.7)")
)

fig.show()


---

### Parte dos

1. Crea una función que identifique aristas inconsistentes.  
2. Revisa todas las aristas que sean inconsistentes y guárdalas.  
3. Una vez identificadas todas las aristas inconsistentes, elimínalas del MST.  
4. Extrae los componentes conexos del resultado del punto 3.  
5. Visualiza las instancias proyectadas, la gráfica resultante del punto 3 y la partición obtenida del punto 4.


In [34]:
#por definición de la presentación
k = 1
q = 2.0

# lineas de MST
L = nx.line_graph(MST)

# cada arista como tupla 
def ord_edge(e):
    u, v = e
    return (u, v) if u < v else (v, u)

# pesos por arista
w = {(min(u, v), max(u, v)): d["weight"] for u, v, d in MST.edges(data=True)}

edges_kept = []      
edges_removed = []   
umbral_por_arista = {}

for e_raw in MST.edges():              
    e = ord_edge(e_raw)                

    # k vecinos 
    neigh_weights = []
    for other, dist in nx.single_source_shortest_path_length(L, e, cutoff=k).items():
        if other == e or dist == 0:
            continue
        neigh_weights.append(w[ord_edge(other)])

    # si no hay vecinas, la conservamos
    if len(neigh_weights) == 0:
        edges_kept.append(e)
        umbral_por_arista[e] = (w[e], 0.0, (w[e], w[e]))
        continue

    neigh = np.asarray(neigh_weights, float)
    mu = neigh.mean()
    sigma = neigh.std(ddof=1)

    if sigma == 0:
        low = high = mu
        is_removed = (w[e] != mu)   
    else:
        low = mu - q * sigma
        high = mu + q * sigma
        is_removed = (w[e] < low) or (w[e] > high)

    (edges_removed if is_removed else edges_kept).append(e)
    umbral_por_arista[e] = (mu, sigma, (low, high))


umbral = float(np.median([thr[2][1] for thr in umbral_por_arista.values()]))



c:\Users\narro\AppData\Local\Programs\Python\Python313\Lib\site-packages\numpy\_core\_methods.py:223: RuntimeWarning:

Degrees of freedom <= 0 for slice

c:\Users\narro\AppData\Local\Programs\Python\Python313\Lib\site-packages\numpy\_core\_methods.py:215: RuntimeWarning:

invalid value encountered in scalar divide



In [35]:

#MDS
random_state = 42
mds = MDS(n_components=2, dissimilarity='precomputed', random_state=random_state)
coords = mds.fit_transform(D)  # shape (n, 2)

#
def xy_from_edges(edges):
    xs, ys = [], []
    for i, j in edges:
        xs.extend([coords[i, 0], coords[j, 0], None])
        ys.extend([coords[i, 1], coords[j, 1], None])
    return xs, ys

x_kept, y_kept = xy_from_edges(edges_kept)
x_removed, y_removed = xy_from_edges(edges_removed)

# 7) Fig de Plotly
fig = go.Figure()

# Líneas del MST conservadas
fig.add_trace(go.Scatter(
    x=x_kept, y=y_kept,
    mode="lines",
    line=dict(width=2),
    opacity=0.9,
    name=f"MST (≤ umbral={umbral:.4g})"
))

show_removed = True
# Líneas removidas (opcional)
if show_removed and len(edges_removed) > 0:
    fig.add_trace(go.Scatter(
        x=x_removed, y=y_removed,
        mode="lines",
        line=dict(width=1, dash="dash"),
        opacity=0.6,
        name=f"Aristas removidas (> {umbral:.4g})"
    ))

# Puntos (instancias)
fig.add_trace(go.Scatter(
    x=coords[:, 0], y=coords[:, 1],
    mode="markers",
    marker=dict(size=10),
    text=labels,
    hovertemplate="<b>%{text}</b><br>X=%{x:.3f}<br>Y=%{y:.3f}<extra></extra>",
    name="Instancias"
))

fig.update_layout(
    title="Proyección 2D (MDS) con MST",
    xaxis_title="X",
    yaxis_title="Y",
    width=800,
    height=700,
    template="plotly_white",
    legend=dict(bgcolor="rgba(255,255,255,0.7)")
)

fig.show()

In [36]:
edges_kept

[(0, 248),
 (1, 379),
 (1, 849),
 (2, 884),
 (2, 397),
 (3, 776),
 (3, 140),
 (4, 181),
 (4, 475),
 (5, 845),
 (5, 882),
 (5, 98),
 (5, 304),
 (6, 151),
 (6, 612),
 (6, 221),
 (7, 993),
 (8, 669),
 (9, 233),
 (9, 718),
 (10, 286),
 (10, 589),
 (10, 576),
 (11, 186),
 (11, 146),
 (12, 292),
 (14, 659),
 (15, 706),
 (16, 173),
 (17, 322),
 (18, 640),
 (18, 53),
 (20, 682),
 (21, 436),
 (21, 698),
 (22, 951),
 (23, 395),
 (23, 221),
 (24, 783),
 (24, 566),
 (24, 285),
 (26, 334),
 (26, 282),
 (26, 753),
 (27, 325),
 (28, 382),
 (28, 329),
 (29, 573),
 (29, 142),
 (30, 948),
 (30, 982),
 (31, 165),
 (31, 797),
 (32, 76),
 (32, 383),
 (33, 965),
 (33, 288),
 (34, 550),
 (34, 84),
 (34, 91),
 (35, 551),
 (36, 832),
 (37, 597),
 (38, 369),
 (38, 893),
 (39, 696),
 (39, 364),
 (40, 594),
 (40, 984),
 (41, 847),
 (41, 729),
 (42, 138),
 (44, 171),
 (44, 507),
 (44, 521),
 (45, 547),
 (45, 764),
 (46, 165),
 (47, 186),
 (48, 487),
 (48, 699),
 (48, 894),
 (48, 278),
 (48, 627),
 (49, 928),
 (49,

In [37]:
edges_removed

[(1, 984),
 (2, 509),
 (12, 418),
 (13, 163),
 (14, 417),
 (18, 862),
 (19, 903),
 (25, 87),
 (32, 702),
 (35, 631),
 (36, 771),
 (37, 155),
 (40, 94),
 (42, 841),
 (43, 440),
 (48, 624),
 (54, 504),
 (56, 445),
 (58, 573),
 (62, 278),
 (65, 124),
 (68, 994),
 (70, 487),
 (71, 788),
 (76, 181),
 (77, 240),
 (78, 319),
 (79, 454),
 (84, 284),
 (88, 836),
 (89, 973),
 (99, 534),
 (102, 434),
 (105, 518),
 (106, 930),
 (107, 191),
 (112, 427),
 (113, 393),
 (114, 321),
 (114, 299),
 (115, 996),
 (116, 866),
 (120, 516),
 (126, 190),
 (130, 215),
 (131, 719),
 (133, 219),
 (140, 308),
 (141, 756),
 (147, 568),
 (150, 179),
 (157, 392),
 (160, 457),
 (164, 385),
 (176, 571),
 (189, 340),
 (195, 232),
 (198, 569),
 (199, 687),
 (200, 498),
 (203, 630),
 (204, 834),
 (207, 299),
 (217, 600),
 (222, 983),
 (233, 235),
 (241, 750),
 (247, 511),
 (252, 801),
 (253, 612),
 (257, 759),
 (259, 922),
 (268, 496),
 (271, 344),
 (273, 446),
 (281, 554),
 (290, 939),
 (293, 562),
 (294, 753),
 (302, 70